In [2]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

# import os
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'


# This is required to run multiple processes with JAX.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from pathlib import Path

# print(jax.devices())
# jax.config.update("jax_debug_nans", True)

from config import Config
import data

In [ ]:
cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn_clip/swot_minimal.yml")
# cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/gcnn/swot_log_mse_no_static.yml")

In [ ]:
cfg.quiet = False
cache_mgr = data.DynamicCacheManager(cfg)
train_cache_dir = cache_mgr.create_cache('train')

In [ ]:
import numpy as np
import xarray as xr
from jaxtyping import Array
from typing import NamedTuple
from tqdm import tqdm

class FlatSample(NamedTuple):
    x_dynamic: Array  # [Features]
    x_static: Array   # [Static Features]
    y: Array          # [Target Features]
    metadata: dict    # basin, subbasin, date

class FlatObservationDataset(data.CachedBasinGraphDataset):
    """
    Fully in-memory flat observation dataset for MLP training.
    Caches dynamic, target, static features and metadata as flat arrays.
    """

    def __init__(self, cfg: Config, cache_dir: Path, subset: str , source_name: str):
        super().__init__(cfg, cache_dir, subset)
        
        if source_name not in self.features.dynamic:
            raise ValueError(f"Source '{source_name}' not found in config dynamic sources.")
        self.source_name = source_name
        self.source_columns = self.features.dynamic[source_name]

        # --- Pre-load everything we need into memory as flat arrays ---
        self._flat_cache = {
            'x_dynamic': [],
            'x_static': [],
            'y': [],
            'metadata': []
        }

        for basin, subbasins in tqdm(self.basin_subset_dict.items(), desc="Caching basins in memory"):
            if not (self.cache_dir / basin).is_dir():
                print(f"{basin} not found in cache")
                continue
                
            with xr.open_zarr(self.cache_dir / basin, consolidated=True) as basin_ds:
                ds = basin_ds.sel(subbasin=basin_ds['subbasin'].isin(list(subbasins))).load()
                
                # Skip basins without required variables
                if not all(col in ds.data_vars for col in self.source_columns):
                    continue
                target_var = self.target[0]
                if target_var not in ds.data_vars:
                    continue
            
                x_dynamic = ds[self.source_columns]  # keep as DataArray for coords
                y = ds[self.target]
            
                valid_features = ~np.isnan(x_dynamic[self.source_columns[0]].values)
                valid_target = ~np.isnan(y.to_array().values[0,...])
                valid_mask = valid_features & valid_target
            
                # Convert to flat indices safely
                t_idxs, s_idxs = np.where(valid_mask)
            
                if len(t_idxs) == 0:
                    continue
            
                # Append to flat cache
                for t_idx, s_idx in zip(t_idxs, s_idxs):
                    self._flat_cache['x_dynamic'].append(x_dynamic.isel(date=t_idx, subbasin=s_idx).to_array().values.astype(np.float32))
                    self._flat_cache['y'].append(y.isel(date=t_idx, subbasin=s_idx).to_array().values.astype(np.float32))
            
                    # Use coordinates directly from the DataArray
                    subbasin_id = str(x_dynamic.subbasin.values[s_idx])
                    date = str(x_dynamic.date.values[t_idx])
                    self._flat_cache['x_static'].append(
                        self.x_s.sel(subbasin=subbasin_id)[self.features.static].to_array().values.astype(np.float32)
                    )
                    self._flat_cache['metadata'].append({
                        'basin': basin,
                        'subbasin': subbasin_id,
                        'date': date
                    })

        # Convert lists to arrays for fast indexing
        self._flat_cache['x_dynamic'] = np.stack(self._flat_cache['x_dynamic'], axis=0)
        self._flat_cache['y'] = np.stack(self._flat_cache['y'], axis=0)
        self._flat_cache['x_static'] = np.stack(self._flat_cache['x_static'], axis=0)
        self.flat_sample_list_len = len(self._flat_cache['x_dynamic'])

        print(f"✅ Cached {self.flat_sample_list_len} valid observation points in memory.")

    def __len__(self):
        return self.flat_sample_list_len

    def __getitem__(self, idx: int):
        return FlatSample(
            x_dynamic=self._flat_cache['x_dynamic'][idx],
            x_static=self._flat_cache['x_static'][idx],
            y=self._flat_cache['y'][idx],
            metadata=self._flat_cache['metadata'][idx]
        )

In [ ]:
from torch.utils.data import DataLoader

def collate_flat_jax(samples):
    basin = np.array([s.metadata['basin'] for s in samples])
    subbasin = np.array([s.metadata['subbasin'] for s in samples])
    date = np.array([s.metadata['date'] for s in samples])
    x_d = jnp.array([s.x_dynamic for s in samples])
    x_s = jnp.array([s.x_static for s in samples])
    y = jnp.array([s.y for s in samples])
    return basin, subbasin, date, x_d, x_s, y

train_dataset = FlatObservationDataset(cfg, train_cache_dir, 'train', 'swot-river')
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=collate_flat_jax)

In [ ]:
sample = train_dataset[0]

d_size = sample.x_dynamic.shape[0]
s_size = sample.x_static.shape[0]

in_size = d_size + s_size
out_size = sample.y.shape[0]

In [ ]:
import equinox as eqx
from models.layers.heads import GMM

key = jax.random.PRNGKey(0)

class MLP_GMM(eqx.Module):
    mlp: eqx.nn.MLP
    head: GMM

    def __init__(self, in_size, hidden_size, depth, n_models, *, key=key):
        k1, k2 = jax.random.split(key)
        self.mlp = eqx.nn.MLP(in_size, hidden_size, hidden_size, depth, key=k1)
        self.head = GMM(hidden_size, hidden_size, n_models, key=k2)

    def __call__(self, x):
        h = self.mlp(x)
        params = self.head(h)

        return params


@eqx.filter_value_and_grad
def compute_gmm_loss(model, x, y):
    params = jax.vmap(model)(x)
    # Extract the params returned by the GMM head
    pi=params["pi"] 
    mu=params["mu"]
    sigma=params["sigma"]

    # Calculate the probability density of y for each Gaussian component.
    one_over_sqrt_2pi = 1.0 / jnp.sqrt(2.0 * jnp.pi)
    exponent = -0.5 * jnp.square((y - mu) / sigma)
    pdf_values = one_over_sqrt_2pi * jnp.exp(exponent) / sigma

    # Weight the PDFs by the mixture weights (pi) and sum them up
    weighted_likelihoods = jnp.sum(pi * pdf_values, axis=-1)

    # negative log likelihood
    nll = -jnp.log(weighted_likelihoods + 1e-6)

    return jnp.mean(nll)



In [ ]:
import jax.numpy as jnp
import equinox as eqx

class McFLI_Net(eqx.Module):
    mlp: eqx.nn.MLP
    
    def __call__(self, swot_vec, prior_q):
        # Concatenate normalized obs + log(prior_q)
        x = jnp.concatenate([swot_vec, jnp.log1p(prior_q)])
        out = self.mlp(x)
        
        # Physics-informed constraints
        # Roughness centered around 0.035
        n = 0.01 + 0.1 * jax.nn.sigmoid(out[0]) 
        # Area scaled by the prior expectation
        A0 = jnp.exp(out[1]) * jnp.sqrt(prior_q) 
        
        return n, A0

def loss_fn(model, batch):
    n, A0 = jax.vmap(model)(batch.swot, batch.prior_q)
    
    # Manning Equation (Vectorized)
    q_pred = (1/n) * (A0 + batch.dA)**(5/3) * batch.W**(-2/3) * batch.S**0.5
    
    # 1. Prediction Loss
    l_q = jnp.mean((jnp.log(q_pred) - jnp.log(batch.q_true))**2)
    
    # 2. Parameter Regularization (The Prior)
    # Penalize n that wanders too far from global median
    l_reg_n = jnp.mean((n - 0.035)**2)
    
    return l_q + 0.1 * l_reg_n

In [ ]:
import optax

key = jax.random.PRNGKey(0)
# model = eqx.nn.MLP(in_size, out_size, width_size=256, depth=6, key=key)
model = MLP_GMM(in_size, 256, 6, 50, key=key)

optim = optax.adam(0.0001)
trainable_model = eqx.filter(model, eqx.is_array)
opt_state = optim.init(trainable_model)


@eqx.filter_jit
def make_step(model, x, y, opt_state):
    loss, grads = compute_gmm_loss(model, x, y)
    updates, opt_state = optim.update(grads, opt_state)
    model = eqx.apply_updates(model, updates)
    return loss, model, opt_state


losses = []
epoch_iter = tqdm(range(10000), desc='Epoch:')
for epoch in epoch_iter:
    e_losses = []
    for batch in train_loader:
        basin, subbasin, date, x_d, x_s, y = batch
        
        x = jnp.concat([x_d, x_s], axis=1)   
        
        loss, model, opt_state = make_step(model, x, y, opt_state)
        e_losses.append(loss.item())
        
    
    loss = np.mean(e_losses)
    losses.append(loss)

    epoch_iter.set_postfix_str(f"{loss = :0.4f}")

    # if epoch % 50 == 0:
    #     print(f"{epoch = }, {loss = :0.4f}")


In [ ]:
plt.plot(losses)

In [ ]:
test_cache_dir = cache_mgr.create_cache('test')
test_dataset = FlatObservationDataset(cfg, test_cache_dir, 'test', 'swot-river')
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=True, collate_fn=collate_flat_jax)

In [ ]:
subbasin_list = []
date_list = []
y_list = []
y_pred_list = []
y_pred_std_list = []

def predict_original_stats(model, x, scale_mean, scale_var):
    params = jax.vmap(model)(x)
    pi, mu_norm, sigma_norm = params["pi"], params["mu"], params["sigma"]

    # 1. Map normalized mu/sigma back to the log(y+1) space
    # mu_log = mu_norm * scale_var + scale_mean
    # sigma_log = sigma_norm * scale_var
    mu_log = (mu_norm * scale_var) + scale_mean
    sigma_log = sigma_norm * scale_var
    var_log = jnp.square(sigma_log)

    # 2. Expected value of each component in original space (y + 1)
    # E[Y+1] = exp(mu_log + 0.5 * var_log)
    comp_mean_orig_plus_1 = jnp.exp(mu_log + 0.5 * var_log)
    
    # 3. Variance of each component in original space
    # Var(Y+1) = [exp(var_log) - 1] * exp(2*mu_log + var_log)
    comp_var_orig = jnp.expm1(var_log) * jnp.exp(2 * mu_log + var_log)

    # 4. Mixture Mean
    y_pred_plus_1 = jnp.sum(pi * comp_mean_orig_plus_1, axis=-1)
    y_pred = y_pred_plus_1 - 1.0 # Subtract the 1 from log1p

    # 5. Mixture Variance (Law of Total Variance)
    total_var_orig = jnp.sum(pi * (comp_var_orig + jnp.square(comp_mean_orig_plus_1)), axis=-1) - jnp.square(y_pred_plus_1)
    y_pred_std = jnp.sqrt(jnp.maximum(total_var_orig, 1e-12))

    return y_pred, y_pred_std

    
scale_mean = test_dataset.d_scale['discharge']['offset']
scale_var = test_dataset.d_scale['discharge']['scale']

for batch in test_loader:
    basin, subbasin, date, x_d, x_s, y = batch
    
    x = jnp.concat([x_d, x_s], axis=1)   
    
    y_pred, y_pred_std = predict_original_stats(model, x, scale_mean, scale_var)

    y = test_dataset.denormalize(y, 'discharge')

    subbasin_list.append(subbasin)
    date_list.append(date)
    y_list.append(y)
    y_pred_list.append(y_pred)
    y_pred_std_list.append(y_pred_std)

pred_data = {
    'subbasin': np.concatenate(subbasin_list),
    'date': np.concatenate(date_list),
    'y': np.concatenate(y_list)[:,0],
    'y_pred': np.concatenate(y_pred_list),
    'y_pred_std': np.concatenate(y_pred_std_list),
}

results = pd.DataFrame(pred_data)

In [ ]:
results

In [ ]:

results.to_parquet('/nas/cee-water/cjgleason/ted/swot-ml/notebooks/multigraph_manual/tests/mlp_gmm_results.parquet')

In [ ]:
results = pd.read_parquet('/nas/cee-water/cjgleason/ted/swot-ml/notebooks/multigraph_manual/tests/mlp_gmm_results.parquet')
results.sort_values('date', inplace=True)

In [ ]:
import evaluate

def group_kge(grp):
    y = grp['y']
    y_hat = grp['y_pred']

    kge_results = evaluate.metrics.calc_kge(y, y_hat)
    if isinstance(kge_results, tuple):
        kge, corr, alpha, beta = kge_results
    else:
        kge = corr = alpha = beta = np.nan

    return kge

kge = results.groupby('subbasin').apply(group_kge, include_groups=False)

In [ ]:
kge

In [ ]:
np.nanmedian(kge)

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))

kge_clean = kge.dropna().values

plt.ecdf(kge_clean)
plt.xlim([-1, 1])
plt.title(f"KGE (test gauges, n = {len(kge)})")

q50 = np.quantile(kge_clean, 0.50)
plt.axvline(q50, color='k', linestyle="--")
plt.text(q50, 0.05, f"50% = {q50:.2f}", rotation=90, va="bottom", ha="right")

q67 = np.quantile(kge_clean, 0.67)
plt.axvline(q67, color='k', linestyle="--")
plt.text(q67, 0.05, f"67% = {q67:.2f}", rotation=90, va="bottom", ha="right")

plt.show()
fig.savefig('/nas/cee-water/cjgleason/ted/swot-ml/notebooks/multigraph_manual/tests/MLP_GMM_KGE.png', dpi=300)

In [ ]:
counts = results.groupby('subbasin').apply(len, include_groups=False).sort_values()

counts[counts>10]

In [ ]:
d = Path("/nas/cee-water/cjgleason/ted/swot-ml/notebooks/multigraph_manual/tests/timeseries")
d.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8,5))

gauge_id = 'USGS-12181000'
gauge_results = results[results['subbasin'] == gauge_id].copy()
gauge_results['date'] = pd.to_datetime(gauge_results['date'])

gauge_results.plot('date','y', ax=ax, label='gauge')
# gauge_results.plot('date','y_pred', ax=ax, label='model')
ax.errorbar(gauge_results['date'], gauge_results['y_pred'], yerr=gauge_results['y_pred_std'], fmt='o', label='model')
ax.set_title(gauge_id)

fig.autofmt_xdate()
fig.savefig(d / f"{gauge_id}.png", dpi=300)

In [ ]:
gauge_results

In [ ]:
import geopandas as gpd 
gauges = gpd.read_parquet("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/metadata/gauges.parquet")
gauges.index = gauges['site_id'].astype(str)

In [ ]:
kge_gdf = gauges.merge(kge.rename('kge').dropna(), left_index=True, right_index=True, how='inner')
kge_gdf

In [ ]:
kge_gdf.plot('kge', vmin=0, vmax=1)